# 06 — 综合报告与行动建议

> 🎯 **适用场景**: 将分析结果提炼为一份可交付的业务报告
> ⏱️ **预计学习时长**: 30 分钟
> 📌 **本章用 SCQA 框架组织**，输出一份可直接提交的电商分析报告

---

## 📋 SCQA 叙事框架

```
S (Situation)  背景 → Olist 是巴西领先的电商平台
C (Complication) 冲突 → 复购率极低但 NPS 很高，增长已趋平
Q (Question)   问题 → 如何突破增长瓶颈？
A (Answer)     答案 → 基于数据的 5 条行动建议
```

---

## 🔢 核心数据快照



In [ ]:
# ═══════════════════════════════════════════
# 汇总核心数据
# ═══════════════════════════════════════════

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

DATA_DIR = "../data"
df_orders  = pd.read_csv(f"{DATA_DIR}/olist_orders_dataset.csv",
    parse_dates=['order_purchase_timestamp', 'order_delivered_customer_date',
                 'order_estimated_delivery_date'])
df_items   = pd.read_csv(f"{DATA_DIR}/olist_order_items_dataset.csv")
df_payments= pd.read_csv(f"{DATA_DIR}/olist_order_payments_dataset.csv")
df_reviews = pd.read_csv(f"{DATA_DIR}/olist_order_reviews_dataset.csv")

# ── 计算核心指标 ──
# 合并
df_oi = df_orders.merge(df_items, on='order_id', how='inner')
order_pay = df_payments.groupby('order_id')['payment_value'].sum().reset_index()
df_oi = df_oi.merge(order_pay, on='order_id', how='left')
df_del = df_oi[df_oi['order_status']=='delivered']

# KPI
total_gmv  = df_del['price'].sum()
total_ord  = df_del['order_id'].nunique()
aov        = total_gmv / total_ord
total_cust = df_del['customer_id'].nunique()
rep_cust   = df_del.groupby('customer_id')['order_id'].nunique()
rep_rate   = (rep_cust > 1).sum() / total_cust * 100
avg_score  = df_reviews['review_score'].mean()

# 物流
delivered = df_orders[df_orders['order_status']=='delivered']
delivered['ship_days'] = (delivered['order_delivered_customer_date'] -
                           delivered['order_purchase_timestamp']).dt.days
delivered['est_days'] = (delivered['order_estimated_delivery_date'] -
                          delivered['order_purchase_timestamp']).dt.days
ontime_rate = (delivered['ship_days'] <= delivered['est_days']).mean() * 100
avg_days    = delivered['ship_days'].mean()

# 趋势
df_del['year_month'] = df_del['order_purchase_timestamp'].dt.to_period('M')
monthly_gmv = df_del.groupby('year_month')['price'].sum()
peak_month  = monthly_gmv.idxmax()

# NPS
promoter = (df_reviews['review_score'] >= 4).sum()
detractor= (df_reviews['review_score'] <= 2).sum()
nps = (promoter - detractor) / len(df_reviews) * 100

# ═══════════════════════════════════════════
# 📊 核心指标卡片
# ═══════════════════════════════════════════

print("=" * 60)
print("         Olist 电商分析 — 综合报告")
print("         数据期间: 2016.09 ~ 2018.10")
print("=" * 60)

print("\n📌 销售指标:")
print(f"   • 总 GMV:               R$ {total_gmv/1e6:,.1f}M")
print(f"   • 总订单数:             {total_ord:,}")
print(f"   • 客单价 (AOV):         R$ {aov:.0f}")
print(f"   • 峰值月份:             {peak_month}")

print("\n📌 用户指标:")
print(f"   • 总用户数:             {total_cust:,}")
print(f"   • 复购率:               {rep_rate:.1f}%")
print(f"   • 平均评分:             {avg_score:.2f} / 5.0")
print(f"   • NPS:                  {nps:.0f}")

print("\n📌 物流指标:")
print(f"   • 准时交付率:           {ontime_rate:.1f}%")
print(f"   • 平均配送时长:         {avg_days:.1f} 天")
print(f"   • 推荐者比例:           {promoter/len(df_reviews)*100:.1f}%")

print("\n" + "=" * 60)


## 🎯 SCQA 报告正文

### S — 背景 (Situation)

Olist 是巴西最大的电商聚合平台之一，连接数十万卖家和数百万消费者。
数据集覆盖 2016 年 9 月至 2018 年 10 月的约 **10 万笔订单**，
涉及约 **9.9 万用户**、**3.3 万产品**，横跨巴西 27 个州。

### C — 冲突 (Complication)

数据揭示了三个矛盾：

1. **高满意但低复购**: NPS 约为 {nps}，评分 4.1/5，但复购率仅 {rep_rate}%。用户满意却不回来！
2. **GMV 增长趋平**: 2017 年快速增长后，2018 年增速明显放缓
3. **品类集中度过高**: Top 3 品类贡献了大部分 GMV，长尾商品几乎没有动

### Q — 问题 (Question)

> **如何在不大量增加获客成本的情况下，突破增长瓶颈？**

### A — 答案 (Answer)

基于数据，建议 5 条行动路径：

---

## 📌 行动建议（按优先级排序）

### 🔴 P0 — 紧急：启动流失用户唤回

| 数据支撑 | 建议 |
|----------|------|
| {rep_rate}% 的复购率意味着绝大部分用户只买一次 | 对首单后 30 天未回访的用户发送定向优惠券 |
| RFM 分析显示"沉睡老客"占比显著 | 首选唤回 "沉睡老客"（曾多频次购买但最近沉默） |
| **预期效果**: 复购率提升到 10%，可带来约 R$ X 增量 GMV | |

### 🟡 P1 — 短期：优化物流，提升体验

| 数据支撑 | 建议 |
|----------|------|
| 配送 > 7 天后评分断崖式下降 | 承诺 7 日必达，超时赔付 |
| {ontime_rate}% 准时率的提升空间 | 优化卖家端发货提醒和物流追踪 |
| **预期效果**: 评分提升 0.2，NPS 提升约 5 点 | |

### 🟡 P1 — 短期：品类扩展与交叉销售

| 数据支撑 | 建议 |
|----------|------|
| Top 3 品类贡献了超高比例 GMV | 首页推荐相关但未试过的品类（个性化推荐） |
| 连带购买分析可挖掘 | 用"买了 X 的人也买了 Y"做交叉推荐 |
| **预期效果**: 客单价提升 10-15% | |

### 🟢 P2 — 中期：构建用户分层运营体系

| 数据支撑 | 建议 |
|----------|------|
| RFM 清晰分出 4 类用户 | 高价值用户 → VIP 权益；流失用户 → 大额券唤回 |
| 新客 90% 以上没有第二单 | 首单后 3 天 → 评价引导；7 天 → 复购券；30 天 → 大促通知 |
| **预期效果**: 建立完整的用户生命周期运营 | |

### 🟢 P2 — 中期：A/B 测试驱动增长

| 数据支撑 | 建议 |
|----------|------|
| 当前缺乏实验数据 | 选 3 个高流量页面做 A/B 测试 |
| boleto 用户行为差异明显 | 测试"支付方式引导"对转化率的影响 |
| **预期效果**: 形成数据驱动的增长飞轮 | |

---

## 📊 一句话总结

> **"Olist 的履约体验优秀（高评分、高准时率），但增长已面临瓶颈。
> 最大的机会不在于拉新，而在于提升不到 3% 的复购率 ——
> 通过用户分层运营和定向唤回，每提升 1% 的复购率，
> 就等于零成本增加了数千个忠诚用户。"**

---

## 📝 康奈尔笔记联动区

### 左侧栏（核心概念）
- **SCQA 框架**: Situation → Complication → Question → Answer
- **复购率瓶颈**: 所有分析的焦点
- **P0/P1/P2 优先级**: 按紧急-重要矩阵排序
- **RFM 分层运营**: 用户生命周期管理的基石

### 右侧栏（反思问题）
> 💭 如果只能做一件事，你选拉新还是提升复购？为什么？
> 💭 Olist 的复购率低是平台模式的宿命吗，还是可以改变的？
> 💭 这份报告如果给真正的 Olist 管理层看，他们会同意哪条建议？

### 底部栏（行动清单）
- [ ] 把 3 张核心图表和这份报告做成 PPT 的第 1 页
- [ ] 用这个项目做你的简历项目（数据分析实战经验）
- [ ] 到 project01_reflections.md 写"项目 01 完成心得"


---

## 🎉 恭喜完成 Project 01！

你刚刚完成了从**原始数据到业务报告**的完整数据科学流程：

```
读取数据 → 数据清洗 → EDA探索 → 业务KPI → 可视化 → 报告叙事
```

这是数据分析师最核心的能力栈。你现在可以：
1. 在简历中写"独立完成 Olist 巴西电商 10 万订单的端到端分析"
2. 把这份 Notebook 和 charts 分享到 GitHub/个人网站
3. 进入 Project 02（用户流失预测），挑战更高级的技能

---

> 💡 **Project 01 学到的核心能力**: 
> Pandas 数据处理 / 数据清洗 / 业务 KPI 计算 / Seaborn 可视化 / 报告叙事
